# `water_vapor()` — `wang-2015`

Estimar PWV a partir do par térmico e do NDVI pelo método Wang.

**Cadeia deste exemplo:** BT10/BT11 + radiância B4/B5 + QA → janelas Wang → PWV.

Este notebook executa uma etapa sobre arquivos já preparados. **Se você tem apenas os arquivos originais Level-1, prepare as entradas indicadas na tabela antes de executar este método.**

Execute apenas os **quatro blocos de código**, de cima para baixo. Os blocos ao final, em “Consulta”, são alternativas para leitura, e não uma continuação obrigatória.


## Antes de executar: separe estes arquivos

| Campo a preencher no passo 1 | Arquivo que você deve escolher | Unidade / tipo | Como obter se ainda falta |
| --- | --- | --- | --- |
| `red_path` | Vermelho B4; por exemplo, `radiancia_B4.tif` | Radiância TOA; W m⁻² sr⁻¹ µm⁻¹ | [Radiância TOA](toa_radiance.ipynb) com `banda=4` |
| `nir_path` | Infravermelho próximo B5; por exemplo, `radiancia_B5.tif` | Radiância TOA; W m⁻² sr⁻¹ µm⁻¹ | [Radiância TOA](toa_radiance.ipynb) com `banda=5` |
| `bt10_path` | Temperatura de brilho B10; por exemplo, `BT_B10_K.tif` | K | [Temperatura de brilho](brightness.ipynb) com `banda=10` |
| `bt11_path` | Temperatura de brilho B11; por exemplo, `BT_B11_K.tif` | K | [Temperatura de brilho](brightness.ipynb) com `banda=11` |
| `qa_pixel_path` | Qualidade dos pixels; por exemplo, `PRODUTO_QA_PIXEL.TIF` | Bits inteiros, sem calibração | Arquivo original QA_PIXEL da mesma cena e na mesma grade |
| `qa_radsat_path` | Saturação/qualidade geométrica; por exemplo, `PRODUTO_QA_RADSAT.TIF` | Bits inteiros, sem calibração | Arquivo original QA_RADSAT da mesma cena e na mesma grade |

Todos os GeoTIFFs devem representar a mesma cena e ter dimensões, CRS e transformação idênticos. Para recortes, use a mesma janela nas QA. As unidades são verificadas pelo usuário: a leitura do arquivo não comprova sua grandeza física.

**Configuração desta execução:** `janela_estimador=21` significa 21 × 21 pixels. A máscara exclui também água neste exemplo terrestre. Os extremos de NDVI são obtidos da área carregada.

**Ambiente:** instale `python -m pip install -e ".[examples]"` na pasta da biblioteca e selecione o kernel desse ambiente no VS Code/Jupyter.


## Passo 1 — Preencher caminhos e escolher a saída

Substitua cada `CAMINHO_...` pelo caminho completo do seu arquivo, mantendo `Path(...)`. No Windows, escreva, por exemplo, `Path(r"C:\dados\radiancia_B4.tif")`. Preencha todos os campos da tabela, inclusive as duas QA.

`janela_estimador=21` significa 21 × 21 pixels. A máscara exclui também água neste exemplo terrestre. Os extremos de NDVI são obtidos da área carregada.

Ajuste `saida` para a pasta desejada. Caminhos relativos partem da pasta de trabalho do kernel; a célula imprime o caminho absoluto. Em nova execução com o mesmo destino, o GeoTIFF é substituído.


In [ ]:
from pathlib import Path
import numpy as np
import rasterio

# EDITE OS CAMINHOS E AS CONFIGURAÇÕES ABAIXO; mantenha o método deste notebook.
from pylstemp import read_quality_mask
from pylstemp import water_vapor

red_path = Path("CAMINHO_DO_VERMELHO_CALIBRADO.tif")
nir_path = Path("CAMINHO_DO_NIR_CALIBRADO.tif")
bt10_path = Path("CAMINHO_DA_BT10_K.tif")
bt11_path = Path("CAMINHO_DA_BT11_K.tif")
qa_pixel_path = Path("CAMINHO_DO_QA_PIXEL.tif")
qa_radsat_path = Path("CAMINHO_DO_QA_RADSAT.tif")

janela_estimador = 21
saida = Path("resultados/water_vapor__wang-2015")

# Confira os arquivos antes de passar ao passo 2.
entradas_obrigatorias = {
    "red_path": red_path,
    "nir_path": nir_path,
    "bt10_path": bt10_path,
    "bt11_path": bt11_path,
    "qa_pixel_path": qa_pixel_path,
    "qa_radsat_path": qa_radsat_path,
}
pendentes = [f"{nome}: {caminho}" for nome, caminho in entradas_obrigatorias.items()
             if not caminho.is_file()]
if pendentes:
    raise FileNotFoundError("Preencha os caminhos de entrada com arquivos existentes:\n"
                            + "\n".join(pendentes))
print("Arquivos encontrados. Próximo passo: ler os dados e aplicar QA.")
print("Pasta de saída:", saida.resolve())


## Passo 2 — Ler as entradas, conferir a grade e aplicar QA

Execute esta célula depois de confirmar os caminhos. Ela carrega os dados, confere seu alinhamento, lê QA_PIXEL/QA_RADSAT com `read_quality_mask()` e combina QA com nodata em `mascara`. `True` indica pixel excluído. A leitura não recalibra dados prontos.

Neste exemplo Wang, a máscara QA também exclui água (`exclude_water=True`). A célula imprime as dimensões e as contagens de exclusão.


In [ ]:
arquivos = {
    "red": red_path,
    "nir": nir_path,
    "bt10": bt10_path,
    "bt11": bt11_path,
}
dados = {}
grade = None
for nome, caminho in arquivos.items():
    with rasterio.open(caminho) as src:
        grade_atual = (src.shape, src.crs, src.transform)
        if grade is None:
            grade = grade_atual
            perfil = src.profile.copy()
        elif grade_atual != grade:
            raise ValueError(f"A imagem {nome} não está na mesma grade das outras entradas.")
        dados[nome] = src.read(1, masked=True).astype(float).filled(np.nan)
red = dados["red"]
nir = dados["nir"]
bt10 = dados["bt10"]
bt11 = dados["bt11"]
mascara = ~np.isfinite(red) | ~np.isfinite(nir) | ~np.isfinite(bt10) | ~np.isfinite(bt11)
# Ler QA na grade da primeira imagem e combinar com nodata antes do método.
mascara_qa = read_quality_mask(
    qa_pixel_path, qa_radsat_path,
    reference_path=next(iter(arquivos.values())),
    exclude_water=True,
)
mascara |= mascara_qa

print("Dimensões:", mascara.shape)
print("Pixels excluídos por QA:", int(mascara_qa.sum()))
print("Pixels excluídos por QA ou nodata:", int(mascara.sum()), "de", mascara.size)


## Passo 3 — Calcular o produto

Execute a chamada abaixo. As variáveis são matrizes carregadas no passo 2; os caminhos de arquivo foram usados no passo 1. `mask=mascara` aplica a seleção QA/nodata ao cálculo.


In [ ]:
resultado = water_vapor(
    method='wang-2015',
    red=red,
    nir=nir,
    brightness_band_10=bt10,
    brightness_band_11=bt11,
    window_size=janela_estimador,
    group_count=3,
    ndvi_range=None,
    chunk_rows=8,
    mask=mascara,
)
imagem = resultado

## Passo 4 — Salvar e usar a saída

Execute esta célula para gravar o GeoTIFF em `saida`, preservando a grade. Confira o caminho absoluto e a quantidade de pixels válidos impressos. Pixels inválidos são gravados como NaN.

**Próximo uso:** Fornecer PWV em g/cm² a Du ou Jiménez SC/SW. O domínio aceito depende do método de temperatura; não converter g/cm² em outra unidade sem explicitar.


In [ ]:
saida.mkdir(parents=True, exist_ok=True)
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=np.nan, compress="deflate")
with rasterio.open(saida / "water_vapor__wang-2015.tif", "w", **perfil) as dst:
    dst.write(np.asarray(imagem, dtype="float32"), 1)
    dst.update_tags(units='g/cm2', method='wang-2015')
print("Pixels válidos:", int(np.isfinite(imagem).sum()), "de", imagem.size)
print("Saída:", saida.resolve())

## Como conferir a execução

Abra o GeoTIFF salvo em um SIG e confira a posição, a unidade e a banda informadas acima. Compare a contagem de pixels válidos com a área disponível: QA pode excluir pixels, e os critérios do próprio método podem produzir lacunas adicionais.

| Se acontecer | O que conferir |
| --- | --- |
| `FileNotFoundError` no passo 1 | Preencha os caminhos listados no erro, inclusive MTL quando solicitado e ambas as QA. |
| Erro de grade no passo 2 | Prepare arquivos com a mesma projeção, transformação, dimensões e recorte. Não redimensione as QA por interpolação contínua. |
| Erro de domínio no passo 3 | Confira a grandeza/unidade e os requisitos do método. |
| Saída com NaN | Confira as contagens QA/nodata e o suporte do método; NaN não deve ser substituído por temperatura inventada. |

[Voltar ao índice e à ordem de execução](README.md).


## Consulta — opções e detalhes do método

O roteiro executável termina acima. Leia os tópicos abaixo para adaptar a entrada ou consultar parâmetros; não execute todas as alternativas em sequência sobre a mesma saída.


### Entradas da função e opções

### Entradas do caminho principal

| Entrada da função | O que fornecer | Unidade / domínio |
|---|---|---|
| `red` | Imagem vermelha calibrada | Radiância TOA, W m⁻² sr⁻¹ µm⁻¹ nesta configuração |
| `nir` | Imagem NIR calibrada | Radiância TOA, W m⁻² sr⁻¹ µm⁻¹ nesta configuração |
| `brightness_band_10` | Temperatura de brilho TOA B10 | K |
| `brightness_band_11` | Temperatura de brilho TOA B11 | K |

**PWV é a saída**, não uma imagem de entrada. O estimador precisa de BT10/BT11 e de NDVI: aqui o NDVI é obtido internamente de vermelho/NIR em radiância TOA. Se tiver NDVI pronto, use `ndvi_image` no lugar das duas bandas.

Todas as imagens usadas juntas devem ter a mesma grade: projeção, resolução, posição dos pixels e dimensões. As variáveis da chamada são arrays de pixels; não são caminhos de arquivo.

### O que é opcional

- **`mask`:** máscara booleana alinhada: True exclui o pixel e False permite avaliar. Opcional na API; neste exemplo é preenchida no passo 2 com QA e nodata. Não passar QA_PIXEL bruta.
- **`window_size`:** janela local, em pixels; padrão 21. As demais opções de qualidade e processamento estão na referência ao final.

### Alternativas de uso

Escolha a alternativa que corresponde aos dados disponíveis; não execute todas as chamadas em sequência sobre a mesma saída.

### 5.1. Fornecer NDVI pronto

Substituir vermelho/NIR por um raster NDVI alinhado, adimensional em [-1,1]. Não fornecer as duas rotas juntas.

```python
imagem = water_vapor(
    method='wang-2015',
    ndvi_image=ndvi_preparado,
    brightness_band_10=bt10,
    brightness_band_11=bt11,
    mask=mascara,
)
```
### Qualidade e processamento

window_size deve ser inteiro ímpar ≥ 3. chunk_rows controla memória temporária, não a equação. Ren usa filtro NDVI local e rejeita controles de grupos globais de Wang; Wang rejeita controles exclusivos de Ren. Veja os detalhes dos parâmetros no final.

### Referência completa dos parâmetros

### Caminho principal

A chamada completa e a tabela abaixo seguem a mesma ordem.

```python
resultado = water_vapor(
    method='wang-2015',
    red=red,
    nir=nir,
    brightness_band_10=bt10,
    brightness_band_11=bt11,
    window_size=janela_estimador,
    group_count=3,
    ndvi_range=None,
    chunk_rows=8,
    mask=mascara,
)
```

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `method` | Obrigatório neste exemplo: selecionar o método | `"wang-2015"` | Seletor | Usar "wang-2015" neste notebook. Os outros métodos têm requisitos próprios e notebooks separados. |
| `red` | Obrigatório na rota por bandas | `None` | Radiância TOA, W m⁻² sr⁻¹ µm⁻¹ nesta configuração | Imagem calibrada e alinhada. Omitir as duas bandas se fornecer ndvi_image preparado. |
| `nir` | Obrigatório na rota por bandas | `None` | Radiância TOA, W m⁻² sr⁻¹ µm⁻¹ nesta configuração | Imagem calibrada e alinhada. Omitir as duas bandas se fornecer ndvi_image preparado. |
| `brightness_band_10` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B10, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `brightness_band_11` | Obrigatório | `Sem padrão` | K | Temperatura de brilho TOA B11, calibrada pelo MTL. Não passar DN, radiância, LST ou Celsius. Mesma forma e grade das demais imagens. |
| `window_size` | Opcional | `None → 21` | Pixels | Lado da janela quadrada, inteiro ímpar ≥3. O suporte em metros depende da resolução da grade; usar halo ao processar blocos. A janela 21 é uma adaptação operacional do suporte 20 do artigo. |
| `group_count` | Opcional somente em Wang | `3` | Contagem | Número inteiro de grupos NDVI ≥1. Não se aplica a Ren. |
| `ndvi_range` | Opcional; global ao processar blocos | `None` | (NDVI mínimo, máximo) | Extremos de toda a cena, obtidos com scene_ndvi_range. Reutilizar em todos os blocos; None determina os extremos da entrada atual. |
| `chunk_rows` | Opcional | `8` | Linhas de pixels | Inteiro ≥1; controla memória/lotes sem mudar as equações. |
| `mask` | Opcional na API; fornecido neste exemplo | `None` | Booleano | Máscara de exclusão alinhada: True rejeita, False permite avaliar. O passo 2 lê ambas as QA e combina seus bits com nodata. Água segue a escolha exclude_water explícita neste notebook. Não fornecer QA bruta; a função pode acrescentar rejeições por domínio. |

### Parâmetros das alternativas

As entradas substituídas e as combinações permitidas são explicadas na consulta de alternativas. Os demais parâmetros permanecem os da tabela principal.

| Parâmetro | Condição | Padrão da API | Unidade | Detalhamento e opções |
|---|---|---|---|---|
| `ndvi_image` | Condicional: alternativa ao par vermelho/NIR | `None` | Adimensional, [-1,1] | NDVI já preparado e alinhado; omitir as duas bandas ópticas. A convenção radiométrica deve respeitar a fonte do método. Não combinar NDVI preparado e vermelho/NIR. |

Os valores escolhidos no exemplo podem diferir do padrão da API; a coluna de padrão descreve a função. As máscaras e as validações de domínio podem produzir NaN, sem preenchimento automático.

### Aplicabilidade e limites

Wang usa janela 21 e três grupos NDVI por padrão. Em blocos, conservar os extremos globais `ndvi_range` da mesma cena e um halo de window_size//2. Com os arquivos completos do exemplo, None calcula os extremos sobre a área carregada; não tratá-los como globais se o arquivo for apenas um recorte.

Exigir máscara apropriada sobre terra, excluindo água e nuvens quando inadequadas ao estimador. Sem amostras, baixa qualidade ou domínio excedido, a saída pode ser NaN; não preencher automaticamente. O domínio do algoritmo de LST que consumir PWV pode ser mais restrito.

Execução e consistência numérica não demonstram acurácia independente em campo. A parametrização deve ser adequada à banda, sensor, superfície e condições da aplicação.

Veja a [ficha metodologica](../docs/method-cards.md) para equacao, coeficientes, dominio e adaptacoes. Execucao numericamente correta nao estabelece acuracia de campo. [Voltar ao indice](README.md).